In [ ]:
import csv
import difflib
import gc
import os
import re
import time
import warnings
from collections import defaultdict, OrderedDict
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import logging

warnings.filterwarnings("ignore", category=DeprecationWarning, module="torch.ao.quantization")
warnings.filterwarnings("ignore", message=".*torch.ao.quantization.*")

# rdflib logs a warning (with traceback) on every malformed literal cast attempt.
logging.getLogger("rdflib").setLevel(logging.ERROR)

import numpy as np
import torch
from sentence_transformers import SentenceTransformer, util

from rdflib import Graph, RDF, RDFS, URIRef, term
from rdflib.namespace import OWL, SKOS

# Don't raise on invalid lexical forms (e.g. malformed xsd:int/xsd:date values).
term._fail_on_invalid_lexical_form = False

# Vocabulary namespaces are schema
_CORE_NAMESPACES = (
    str(RDF),
    str(RDFS),
    str(OWL),
    str(SKOS),
)


class MOSAIC:
    _CAMEL_RE = re.compile(r'([a-z])([A-Z])')
    _PCT_RE = re.compile(r'%[0-9A-Fa-f]{2}')

    DEFAULT_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

    def __init__(self, model_name=None, thresholds=None, max_cache_labels=400_000):
        # Per-entity-type similarity thresholds; tuned for the default model's score distribution.
        self.thresholds = thresholds or {
            OWL.Class: 0.80,
            SKOS.Concept: 0.80,
            OWL.ObjectProperty: 0.88,
            OWL.DatatypeProperty: 0.88,
            OWL.NamedIndividual: 0.82
        }
        self.default_thres = 0.80
        self.model = None
        self.model_name = model_name or self.DEFAULT_MODEL

        self.entity_cache = {}                 # path -> extracted entities
        self.string_emb_cache = OrderedDict()  # label -> embedding tensor, LRU-bounded
        self.max_cache_labels = max_cache_labels

        # Leave one core free for the OS/other work.
        cores = os.cpu_count() or 1
        if hasattr(os, 'sched_getaffinity'):
            try:
                cores = len(os.sched_getaffinity(0))
            except Exception:
                pass
        threads = max(4, cores - 1)
        torch.set_num_threads(threads)

    def init_model(self):
        # Lazily loads the embedding model and picks int8-quantized vs full precision, whichever benchmarks faster.
        if self.model is None:
            print(f" [MOSAIC] Initializing embedding engine on CPU: {self.model_name}")
            raw_model = SentenceTransformer(self.model_name, device="cpu")

            try:
                raw_model.max_seq_length = 64
            except Exception:
                pass

            probe = [f"sample ontology label {i} for benchmarking throughput" for i in range(64)]

            t0 = time.time()
            with torch.inference_mode():
                raw_model.encode(probe, convert_to_tensor=True, show_progress_bar=False, batch_size=64)
            raw_time = time.time() - t0

            quantized_model = None
            quant_time = None
            try:
                from torch.ao.quantization import quantize_dynamic
                quantized_model = quantize_dynamic(raw_model, {torch.nn.Linear}, dtype=torch.qint8, inplace=False)
                t0 = time.time()
                with torch.inference_mode():
                    quantized_model.encode(probe, convert_to_tensor=True, show_progress_bar=False, batch_size=64)
                quant_time = time.time() - t0
            except Exception as e:
                print(f" [NOTICE] Quantization unavailable ({e}). Running model at standard precision.")

            if quantized_model is not None and quant_time < raw_time:
                self.model = quantized_model
                print(f" [SUCCESS] INT8 quantization measured faster ({quant_time:.3f}s vs {raw_time:.3f}s) — using it.")
            else:
                self.model = raw_model
                if quantized_model is not None:
                    print(f" [NOTICE] Quantization measured slower here ({quant_time:.3f}s vs {raw_time:.3f}s) — keeping full precision.")

            del probe, quantized_model

    def clear_caches(self):
        # Drop cached entities/embeddings between tracks so memory doesn't accumulate.
        self.entity_cache.clear()
        self.string_emb_cache.clear()
        gc.collect()

    def normalise_label(self, text: str) -> str:
        # Splits camelCase, strips separators, lowercases, and caps length at 20 words.
        if not text:
            return ""
        text = self._CAMEL_RE.sub(r'\1 \2', str(text))
        text = text.replace('_', ' ').replace('-', ' ').lower().strip()
        words = text.split()
        return " ".join(words[:20])

    def get_text_data(self, uri, graph) -> str:
        # Picks the best available label for a URI, preferring target languages, then falls back to the URI fragment.
        target_langs = ['de', 'fr', 'sl', 'hr', 'en', 'ar', 'es', 'it', 'nl']
        for lang in target_langs:
            for label in graph.objects(uri, SKOS.prefLabel):
                if hasattr(label, 'language') and label.language == lang:
                    return self.normalise_label(label)
            for label in graph.objects(uri, RDFS.label):
                if hasattr(label, 'language') and label.language == lang:
                    return self.normalise_label(label)

        label = graph.value(uri, RDFS.label) or graph.value(uri, SKOS.prefLabel)
        if label:
            return self.normalise_label(label)

        frag = str(uri).split('/')[-1].split('#')[-1]
        frag = self._PCT_RE.sub(' ', frag)
        return self.normalise_label(frag)

    def load_ontology(self, path: Path) -> Graph:
        # Tries likely serialization formats based on file extension before falling back to auto-detect.
        g = Graph()
        formats = ["turtle", "xml"] if path.suffix == ".ttl" else (["xml", "turtle"] if path.suffix in [".owl", ".rdf", ".xml"] else [None])
        for fmt in formats:
            try:
                g.parse(str(path), format=fmt)
                return g
            except Exception:
                continue
        try:
            g.parse(str(path))
            return g
        except Exception as e:
            print(f" [MOSAIC] Failed to load {path.name}: {e}")
            return None

    def get_entities_cached(self, path: Path, graph: Graph):
        # Returns extracted entities for a path, computing and caching them on first access.
        if path not in self.entity_cache:
            self.entity_cache[path] = self.extract_entities(graph)
        return self.entity_cache[path]

    def extract_entities(self, graph: Graph):
        # Buckets graph subjects into classes/concepts/properties/individuals and attaches normalized labels.
        skos_concepts = set(graph.subjects(RDF.type, SKOS.Concept))
        owl_classes = set(graph.subjects(RDF.type, OWL.Class)) - skos_concepts
        props = set(graph.subjects(RDF.type, OWL.ObjectProperty)).union(set(graph.subjects(RDF.type, OWL.DatatypeProperty)))
        all_subs = set(graph.subjects())
        insts = all_subs - skos_concepts - owl_classes - props

        def is_real_entity(uri):
            if not isinstance(uri, URIRef):
                return False
            s = str(uri)
            if "oboInOwl" in s:
                return False
            if s.startswith(_CORE_NAMESPACES):
                return False
            return True

        insts = {i for i in insts if is_real_entity(i)}
        owl_classes = {c for c in owl_classes if is_real_entity(c)}
        skos_concepts = {c for c in skos_concepts if is_real_entity(c)}
        props = {p for p in props if is_real_entity(p)}

        entities = {}
        for entity_set, etype in [(owl_classes, OWL.Class), (skos_concepts, SKOS.Concept), (props, OWL.ObjectProperty), (insts, OWL.NamedIndividual)]:
            for s in entity_set:
                lbl_str = self.get_text_data(s, graph)
                entities[s] = {
                    "label": lbl_str,
                    "tokens": lbl_str.split(),
                    "type": etype,
                }
        return entities

    def get_embeddings_granular(self, labels: list, batch_size=2048):
        # Encodes only labels missing from cache, then returns stacked embeddings for all requested labels.
        missing_labels = list(dict.fromkeys(
            lbl for lbl in labels if lbl not in self.string_emb_cache
        ))

        if missing_labels:
            self.init_model()
            encoded_missing = self.model.encode(
                missing_labels, convert_to_tensor=True, show_progress_bar=False, batch_size=batch_size
            )
            for lbl, tensor in zip(missing_labels, encoded_missing):
                self.string_emb_cache[lbl] = tensor.cpu()
            del encoded_missing

        # LRU eviction: touch requested labels, then trim any overflow past max_cache_labels.
        for lbl in labels:
            self.string_emb_cache.move_to_end(lbl)
        while len(self.string_emb_cache) > self.max_cache_labels:
            self.string_emb_cache.popitem(last=False)

        return torch.stack([self.string_emb_cache[lbl] for lbl in labels])

    def build_inverted_index(self, labels, tokens_list):
        # Builds a word -> label-indices index, skipping overly common words to keep candidate sets small.
        index = defaultdict(list)
        counts = defaultdict(int)
        for tokens in tokens_list:
            for word in set(tokens):
                if len(word) > 2:
                    counts[word] += 1

        max_limit = max(15, int(len(labels) * 0.01))
        for idx, tokens in enumerate(tokens_list):
            for word in tokens:
                if len(word) > 2 and counts[word] <= max_limit:
                    index[word].append(idx)
        return index

    def fast_token_match(self, src_tokens, tgt_labels, tgt_tokens_list, inverted_index):
        # Cheap fallback matcher: finds the target label with the most shared tokens.
        if not src_tokens: return 0.0, 0
        counts = defaultdict(int)
        for word in src_tokens:
            if word in inverted_index:
                for idx in inverted_index[word]:
                    counts[idx] += 1
        if not counts: return 0.0, 0

        best_idx = max(counts, key=counts.get)
        match_count = counts[best_idx]
        max_words = max(len(src_tokens), len(tgt_tokens_list[best_idx]))
        return (match_count / max_words if max_words > 0 else 0.0), best_idx

    def semantic_similarity_by_type(self, filtered_src, filtered_tgt, etype, batch_size=2048, chunk_size=8192):
        # Finds best-match candidates between src/tgt entities of one type using embeddings, with token-blocking for large sets.
        src_subset = [ (uri, meta) for uri, meta in filtered_src.items() if meta["type"] == etype ]
        tgt_subset = [ (uri, meta) for uri, meta in filtered_tgt.items() if meta["type"] == etype ]
        if not src_subset or not tgt_subset:
            return []

        required_threshold = self.thresholds.get(etype, self.default_thres)

        src_uris, src_metas = zip(*src_subset)
        tgt_uris, tgt_metas = zip(*tgt_subset)

        src_labels = [m["label"] for m in src_metas]
        src_tokens_list = [m["tokens"] for m in src_metas]
        tgt_labels = [m["label"] for m in tgt_metas]
        tgt_tokens_list = [m["tokens"] for m in tgt_metas]

        tgt_index = self.build_inverted_index(tgt_labels, tgt_tokens_list)

        with torch.inference_mode():
            emb1 = self.get_embeddings_granular(src_labels, batch_size=batch_size)
            emb2 = self.get_embeddings_granular(tgt_labels, batch_size=batch_size)

        n_src, n_tgt = len(src_labels), len(tgt_labels)
        use_blocking = n_tgt > 150
        candidates = []

        with torch.inference_mode():
            for start in range(0, n_src, chunk_size):
                end = min(start + chunk_size, n_src)

                chunk_candidates = []
                union_cols = []
                col_pos = None
                emb2_sub = None

                if use_blocking:
                    for s_tokens in src_tokens_list[start:end]:
                        cols = set()
                        for w in s_tokens:
                            if w in tgt_index:
                                cols.update(tgt_index[w])
                        chunk_candidates.append(cols)

                    # Only score against candidate columns (token overlap), not the full target set.
                    union_col_set = set()
                    for cols in chunk_candidates:
                        union_col_set.update(cols)

                    if not union_col_set:
                        continue

                    union_cols = sorted(union_col_set)
                    col_pos = {c: p for p, c in enumerate(union_cols)}
                    emb2_sub = emb2[union_cols]
                    sim_chunk = util.cos_sim(emb1[start:end], emb2_sub)
                else:
                    sim_chunk = util.cos_sim(emb1[start:end], emb2)

                for local_i in range(end - start):
                    i = start + local_i
                    s_lbl = src_labels[i]
                    s_tokens = src_tokens_list[i]

                    if use_blocking:
                        allowed_cols = chunk_candidates[local_i]
                        if not allowed_cols:
                            continue
                        local_cols = [col_pos[c] for c in allowed_cols]
                        row_sims = sim_chunk[local_i, local_cols]
                        if row_sims.numel() == 0:
                            continue
                        local_best = torch.argmax(row_sims).item()
                        best_tgt_idx = union_cols[local_cols[local_best]]
                        score = row_sims[local_best].item()
                    else:
                        score, best_tgt_idx = torch.max(sim_chunk[local_i], dim=0)
                        score = score.item()

                    t_lbl = tgt_labels[best_tgt_idx]

                    if use_blocking:
                        # Penalize embedding score when string similarity is low and the score is borderline.
                        if abs(len(s_lbl) - len(t_lbl)) > max(len(s_lbl), len(t_lbl)) * 0.5:
                            ratio = 0.0
                        else:
                            ratio = difflib.SequenceMatcher(None, s_lbl, t_lbl).quick_ratio()

                        if ratio < 0.45 and score < (required_threshold + 0.05):
                            score *= 0.75

                    if score < required_threshold:
                        t_score, fast_idx = self.fast_token_match(s_tokens, tgt_labels, tgt_tokens_list, tgt_index)
                        if t_score > 0.65 and t_score > score:
                            score = t_score
                            best_tgt_idx = fast_idx

                    if score >= required_threshold:
                        candidates.append({
                            "source": src_uris[i],
                            "target": tgt_uris[best_tgt_idx],
                            "type": etype,
                            "combined_score": score
                        })

                del sim_chunk
                if emb2_sub is not None:
                    del emb2_sub

        del emb1, emb2
        return candidates

    def align(self, src_graph: Graph, tgt_graph: Graph, src_path: Path, tgt_path: Path, preferred_skos_pred: str = "http://www.w3.org/2002/07/owl#sameAs"):
        # Aligns two ontologies: exact label matches first, then semantic matching for the rest.
        t_extract_start = time.time()
        src_ents = self.get_entities_cached(src_path, src_graph)
        tgt_ents = self.get_entities_cached(tgt_path, tgt_graph)
        extract_time = round(time.time() - t_extract_start, 2)
        print(f"   [MOSAIC] Entity extraction: {extract_time}s ({len(src_ents)} src / {len(tgt_ents)} tgt entities)")

        final_pool = []
        claimed_src = set()
        claimed_tgt = set()

        tgt_lookup = {meta["label"]: uri for uri, meta in tgt_ents.items() if meta["label"]}

        for s_uri, s_meta in src_ents.items():
            s_lbl = s_meta["label"]
            if s_lbl in tgt_lookup:
                t_uri = tgt_lookup[s_lbl]
                t_meta = tgt_ents[t_uri]
                if s_meta["type"] == t_meta["type"]:
                    claimed_src.add(s_uri)
                    claimed_tgt.add(t_uri)
                    final_pool.append({
                        "source": s_uri,
                        "target": t_uri,
                        "type": s_meta["type"],
                        "combined_score": 1.0
                    })

        filtered_src = {k: v for k, v in src_ents.items() if k not in claimed_src}
        filtered_tgt = {k: v for k, v in tgt_ents.items() if k not in claimed_tgt}

        if filtered_src and filtered_tgt:
            # Embed each graph's full label set once instead of per-type, so length-sorting/batching is more effective.
            t_embed_start = time.time()
            all_src_labels = list({m["label"] for m in filtered_src.values() if m["label"]})
            all_tgt_labels = list({m["label"] for m in filtered_tgt.values() if m["label"]})
            with torch.inference_mode():
                if all_src_labels:
                    self.get_embeddings_granular(all_src_labels)
                if all_tgt_labels:
                    self.get_embeddings_granular(all_tgt_labels)
            embed_time = round(time.time() - t_embed_start, 2)

            distinct_types = [OWL.Class, SKOS.Concept, OWL.ObjectProperty, OWL.DatatypeProperty, OWL.NamedIndividual]
            sem_candidates = []
            t_match_start = time.time()
            for etype in distinct_types:
                sem_candidates.extend(self.semantic_similarity_by_type(filtered_src, filtered_tgt, etype))
            match_time = round(time.time() - t_match_start, 2)
            print(f"   [MOSAIC] Embedding: {embed_time}s ({len(all_src_labels) + len(all_tgt_labels)} unique labels) | Matching: {match_time}s")

            sorted_pairs = sorted(sem_candidates, key=lambda x: x["combined_score"], reverse=True)
            for c in sorted_pairs:
                if c["source"] not in claimed_src and c["target"] not in claimed_tgt:
                    claimed_src.add(c["source"])
                    claimed_tgt.add(c["target"])
                    final_pool.append(c)

        alignments = set()
        eq_class = "http://www.w3.org/2002/07/owl#equivalentClass"
        eq_prop = "http://www.w3.org/2002/07/owl#equivalentProperty"
        same_as = "http://www.w3.org/2002/07/owl#sameAs"

        for c in final_pool:
            s_uri, t_uri, etype = c["source"], c["target"], c["type"]
            if etype == OWL.Class:
                alignments.add((str(s_uri), eq_class, str(t_uri)))
            elif etype == SKOS.Concept:
                alignments.add((str(s_uri), preferred_skos_pred, str(t_uri)))
            elif etype in [OWL.ObjectProperty, OWL.DatatypeProperty]:
                alignments.add((str(s_uri), eq_prop, str(t_uri)))
            else:
                alignments.add((str(s_uri), same_as, str(t_uri)))

        return alignments


class OAEITrackRunner:

    def __init__(self, matcher: MOSAIC):
        self.matcher = matcher
        self.log = []

    def load_reference_alignments(self, path: Path) -> set:
        # Parses a reference TTL file into a set of (subject, predicate, object) triples.
        ref_set = set()
        g = Graph()
        try:
            g.parse(str(path), format="turtle")
            valid_preds = {
                "http://www.w3.org/2002/07/owl#equivalentClass",
                "http://www.w3.org/2000/01/rdf-schema#subClassOf",
                "http://www.w3.org/2002/07/owl#equivalentProperty",
                "http://www.w3.org/2000/01/rdf-schema#subPropertyOf",
                "http://www.w3.org/2002/07/owl#sameAs",
            }
            for s, p, o in g:
                if str(p) in valid_preds:
                    nodes = sorted([str(s), str(o)])
                    ref_set.add((nodes[0], str(p), nodes[1]))
        except Exception as e:
            print(f" Could not read reference file {path.name}: {e}")
        return ref_set

    def serialize_alignments_to_ttl(self, alignments: set, path: Path):
        # Writes alignment triples to disk as turtle.
        g = Graph()
        for src, pred, tgt in alignments:
            g.add((URIRef(src), URIRef(pred), URIRef(tgt)))
        try:
            g.serialize(destination=str(path), format="turtle")
            print(f"   [MOSAIC] Output saved to: {path.parent.name}/{path.name}")
        except Exception as e:
            print(f"   [MOSAIC] Serialization error: {e}")

    def calculate_metrics(self, sys_align, ref_align):
        # Computes precision/recall/F1 of system alignments against a reference set.
        if not ref_align:
            return 0.0, 0.0, 0.0
        sys_canon = set()
        for s, p, o in sys_align:
            nodes = sorted([str(s), str(o)])
            sys_canon.add((nodes[0], str(p), nodes[1]))

        tp = len(sys_canon.intersection(ref_align))
        p = tp / len(sys_canon) if sys_canon else 0.0
        r = tp / len(ref_align) if ref_align else 0.0
        f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0.0
        return round(p, 4), round(r, 4), round(f1, 4)

    def find_ontology_file(self, folder: Path, name: str) -> Path:
        # Locates an ontology file by name, trying common extensions in order.
        for ext in [".owl", ".rdf", ".ttl", ".xml"]:
            p = folder / f"{name}{ext}"
            if p.exists():
                return p
        return None

    def calibrate_thresholds(self, src_graph, tgt_graph, src_path, tgt_path, ref_align,
                              grid=None, preferred_skos_pred="http://www.w3.org/2002/07/owl#sameAs"):
        # Coordinate-descent sweep per entity type to find thresholds maximizing F1 against a reference.
        # Calibrate on a held-out task; using the same task for reporting inflates F1 (resubstitution bias).
        if grid is None:
            grid = [round(v, 2) for v in np.arange(0.55, 0.96, 0.05)]

        types = [OWL.Class, SKOS.Concept, OWL.ObjectProperty, OWL.DatatypeProperty, OWL.NamedIndividual]
        best_thresholds = dict(self.matcher.thresholds)

        src_ents = self.matcher.get_entities_cached(src_path, src_graph)
        tgt_ents = self.matcher.get_entities_cached(tgt_path, tgt_graph)
        src_type_counts = defaultdict(int)
        tgt_type_counts = defaultdict(int)
        for m in src_ents.values():
            src_type_counts[m["type"]] += 1
        for m in tgt_ents.values():
            tgt_type_counts[m["type"]] += 1

        print("   [CALIBRATE] Sweeping per-type thresholds against reference alignment...")
        t_cal_start = time.time()
        for etype in types:
            short_name = str(etype).split('#')[-1]
            if src_type_counts.get(etype, 0) == 0 or tgt_type_counts.get(etype, 0) == 0:
                print(f"   [CALIBRATE] {short_name}: skipped (absent on one side, {src_type_counts.get(etype, 0)} src / {tgt_type_counts.get(etype, 0)} tgt)")
                continue

            best_f1 = -1.0
            best_val = best_thresholds.get(etype, self.matcher.default_thres)
            for val in grid:
                trial = dict(best_thresholds)
                trial[etype] = val
                self.matcher.thresholds = trial
                alignments = self.matcher.align(
                    src_graph, tgt_graph, src_path, tgt_path, preferred_skos_pred=preferred_skos_pred
                )
                _, _, f1 = self.calculate_metrics(alignments, ref_align)
                if f1 > best_f1:
                    best_f1 = f1
                    best_val = val
            best_thresholds[etype] = float(best_val)
            print(f"   [CALIBRATE] {short_name}: threshold={best_val} (F1={best_f1})")

        cal_time = round(time.time() - t_cal_start, 2)
        self.matcher.thresholds = best_thresholds
        readable = {str(k).split('#')[-1]: v for k, v in best_thresholds.items()}
        print(f"   [CALIBRATE] Done in {cal_time}s. Final thresholds: {readable}")
        return best_thresholds

    def run_all_tracks(self, base_dir: str, csv_out: str = "mosaic_evaluation_report.csv", calibrate: bool = False):
        # Runs alignment across every track/task under base_dir, logging metrics and writing a CSV report.
        start_global = time.time()
        base_path = Path(base_dir)
        res_dir = Path("../results")
        res_dir.mkdir(parents=True, exist_ok=True)

        if not base_path.exists():
            print(f"Error: Base directory '{base_dir}' does not exist.")
            return

        for track in base_path.iterdir():
            if not track.is_dir():
                continue

            print(f"\n" + "=" * 50)
            print(f" TRACK RUNNER: {track.name.upper()}")
            print(f"=" * 50)

            tasks = list(track.glob("*.ttl"))
            p_sum, r_sum, f_sum, t_sum = 0.0, 0.0, 0.0, 0.0
            count = 0
            calibrated_this_track = False

            for tf in tasks:
                parts = tf.stem.split("-")
                if len(parts) != 2:
                    if "human-mouse" in tf.stem:
                        parts = ["human", "mouse"]
                    else:
                        continue

                ont_folder = track / "ontologies"
                src_p = self.find_ontology_file(ont_folder, parts[0])
                tgt_p = self.find_ontology_file(ont_folder, parts[1])

                print(f"\nMOSAIC Task: {parts[0]} ➔ {parts[1]}")

                if not src_p or not tgt_p:
                    print(" Skipping task. Missing ontology file.")
                    continue

                ref_align = self.load_reference_alignments(tf)
                preferred_skos_pred = "http://www.w3.org/2002/07/owl#sameAs"
                for _, p, _ in ref_align:
                    if "equivalentClass" in p:
                        preferred_skos_pred = "http://www.w3.org/2002/07/owl#equivalentClass"
                        break

                with ThreadPoolExecutor(max_workers=2) as executor:
                    future_src = executor.submit(self.matcher.load_ontology, src_p)
                    future_tgt = executor.submit(self.matcher.load_ontology, tgt_p)
                    src_g = future_src.result()
                    tgt_g = future_tgt.result()

                is_calibration_task = False
                if calibrate and not calibrated_this_track and ref_align and src_g and tgt_g:
                    print(" [CALIBRATE] Using this task to tune thresholds for the current model "
                          "— its own reported F1 below will be optimistic (tuned on itself).")
                    self.calibrate_thresholds(src_g, tgt_g, src_p, tgt_p, ref_align,
                                               preferred_skos_pred=preferred_skos_pred)
                    calibrated_this_track = True
                    is_calibration_task = True

                if src_g and tgt_g:
                    t0 = time.time()
                    alignments = self.matcher.align(src_g, tgt_g, src_p, tgt_p, preferred_skos_pred=preferred_skos_pred)
                    dt = round(time.time() - t0, 2)

                    print(f" Step complete. MOSAIC returned {len(alignments)} matches in {dt}s.")

                    out_ttl = res_dir / f"mosaic_{track.name}_{tf.name}"
                    self.serialize_alignments_to_ttl(alignments, out_ttl)

                    p, r, f1 = self.calculate_metrics(alignments, ref_align)
                    print(f"   Metrics -> Precision: {p}, Recall: {r}, F1-Score: {f1} (Time: {dt}s)")

                    self.log.append({
                        "Track": track.name,
                        "Task": tf.stem,
                        "Precision": p,
                        "Recall": r,
                        "F1-Score": f1,
                        "Time (s)": dt,
                        "Type": "Task (calibration — optimistic)" if is_calibration_task else "Task",
                    })

                    p_sum += p
                    r_sum += r
                    f_sum += f1
                    t_sum += dt
                    count += 1

                    del src_g, tgt_g
                    gc.collect()

            if count > 0:
                avg_p = round(p_sum / count, 4)
                avg_r = round(r_sum / count, 4)
                avg_f1 = round(f_sum / count, 4)
                avg_t = round(t_sum / count, 2)

                print(f"\n Track [{track.name}] AVERAGES -> P: {avg_p}, R: {avg_r}, F1: {avg_f1} | Avg Time: {avg_t}s")

                self.log.append({
                    "Track": track.name,
                    "Task": "TRACK_AVERAGE",
                    "Precision": avg_p,
                    "Recall": avg_r,
                    "F1-Score": avg_f1,
                    "Time (s)": avg_t,
                    "Type": "Average",
                })

            # Tracks don't share ontologies, so clear caches before the next one.
            self.matcher.clear_caches()

        total_runtime = round(time.time() - start_global, 2)
        print(f"\n" + "=" * 50)
        print(f" RUN COMPLETION: Finished in {total_runtime}s.")
        print(f"=" * 50)

        self.log.append({
            "Track": "ALL_TRACKS",
            "Task": "TOTAL_PROGRAM_TIME",
            "Precision": "",
            "Recall": "",
            "F1-Score": "",
            "Time (s)": total_runtime,
            "Type": "Summary",
        })
        self.results_to_csv(csv_out)

    def results_to_csv(self, filename: str):
        # Writes the accumulated run log to a CSV file.
        fields = ["Track", "Task", "Precision", "Recall", "F1-Score", "Time (s)", "Type"]
        with open(filename, mode="w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fields)
            writer.writeheader()
            writer.writerows(self.log)
        print(f" Compilation written to: {filename}")


if __name__ == "__main__":
    custom_thresholds = {
        OWL.Class: 0.75,
        SKOS.Concept: 0.9,
        OWL.ObjectProperty: 0.8,
        OWL.DatatypeProperty: 0.56,
        OWL.NamedIndividual: 0.8
    }

    m = MOSAIC(
        model_name="sentence-transformers/LaBSE",
        thresholds=custom_thresholds,
        max_cache_labels=400_000,
    )
    runner = OAEITrackRunner(matcher=m)
    # Set calibrate=True to auto-tune thresholds per track
    runner.run_all_tracks("../tracks", csv_out="mosaic_evaluation_report.csv", calibrate=False)